# Operating Systems for Everyday Debugging

Notebook for **Chapter 5** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Processes, Threads, Context Switches and Scheduling

In [ ]:
from collections import deque
def fcfs(jobs):
    t=0; res={}
    for n,b in jobs:
        t+=b; res[n]=t
    return res
def rr(jobs,q):
    dq=deque(jobs); t=0; res={}
    while dq:
        n,b=dq.popleft()
        run=min(q,b); t+=run; b-=run
        if b: dq.append((n,b))
        else: res[n]=t
    return res
jobs=[("A",24),("B",3),("C",3)]
for name,r in [("FCFS",fcfs(jobs)),("RR q=4",rr(jobs,4))]:
    print(name,r,"avg finish",round(sum(r.values())/len(r),2))

In [ ]:
import time, multiprocessing as mp, threading, os
def burn(n=3_000_000):
    s=0
    for i in range(n): s+=i*i
    return s
if __name__=="__main__":
    print("cores:", os.cpu_count())
    t=time.perf_counter(); [burn() for _ in range(4)]; print(f"serial x4: {time.perf_counter()-t:.2f}s")
    t=time.perf_counter()
    ts=[threading.Thread(target=burn) for _ in range(4)]
    [x.start() for x in ts]; [x.join() for x in ts]
    print(f"4 threads: {time.perf_counter()-t:.2f}s")
    t=time.perf_counter()
    with mp.Pool(4) as p: p.map(burn,[3_000_000]*4)
    print(f"4 processes: {time.perf_counter()-t:.2f}s")

## 5.2 Concurrency Problems: Races, Deadlocks, Starvation, Locks

In [ ]:
import threading, time
counter = 0
def work_unsafe(n):
    global counter
    for _ in range(n):
        tmp = counter
        time.sleep(0)
        counter = tmp + 1
lock = threading.Lock()
def work_safe(n):
    global counter
    for _ in range(n):
        with lock:
            tmp = counter
            time.sleep(0)
            counter = tmp + 1
def run(fn, label, threads=4, n=2000):
    global counter
    counter = 0
    ts = [threading.Thread(target=fn, args=(n,)) for _ in range(threads)]
    for t in ts: t.start()
    for t in ts: t.join()
    print(f"{label}: expected {threads*n}, got {counter}")
run(work_unsafe, "no lock  ")
run(work_safe,   "with lock")

In [ ]:
import threading, time
a, b = threading.Lock(), threading.Lock()
def t1():
    with a:
        time.sleep(0.2)
        ok = b.acquire(timeout=1)
        print("t1 got b?", ok)
        if ok: b.release()
def t2():
    with b:
        time.sleep(0.2)
        ok = a.acquire(timeout=1)
        print("t2 got a?", ok)
        if ok: a.release()
def t2_fixed():
    with a:
        time.sleep(0.2)
        with b:
            print("t2 got both")

## 5.3 Memory: Stack, Heap, Virtual Memory, Swapping, OOM

In [ ]:
import tracemalloc, sys
cache = []
def handle(i):
    cache.append("x"*10_000)   # leak: never cleared
tracemalloc.start()
for i in range(1000): handle(i)
cur, peak = tracemalloc.get_traced_memory()
print(f"after 1000 requests: {cur/1e6:.1f} MB held")
for i in range(1000): handle(i)
cur, peak = tracemalloc.get_traced_memory()
print(f"after 2000 requests: {cur/1e6:.1f} MB held")
print("list of 1000 ints:", sys.getsizeof(list(range(1000))), "bytes (pointers only)")
def rec(n): return rec(n+1)
try: rec(0)
except RecursionError as e: print("stack overflow guard:", type(e).__name__)

## 5.4 File Systems and I/O Models

In [ ]:
import asyncio, time
def blocking():
    t=time.perf_counter()
    for _ in range(5): time.sleep(0.2)
    print(f"blocking sequential: {time.perf_counter()-t:.1f}s")
async def one(): await asyncio.sleep(0.2)
async def main():
    t=time.perf_counter()
    await asyncio.gather(*[one() for _ in range(5)])
    print(f"asyncio, 5 waits together: {time.perf_counter()-t:.1f}s")
blocking(); asyncio.run(main())

In [ ]:
import os, time, tempfile
d=tempfile.mkdtemp(dir="/tmp"); p=os.path.join(d,"f.bin")
with open(p,"wb") as f: f.write(os.urandom(200_000_000))
for label in ["first read","second read"]:
    t=time.perf_counter()
    with open(p,"rb") as f:
        while f.read(1<<20): pass
    print(f"{label}: {time.perf_counter()-t:.3f}s")
os.remove(p); os.rmdir(d)

## 5.5 Diagnosing a Slow System on Linux

## 5.6 Containers from the Inside: Namespaces, cgroups and Exit Code 137

In [ ]:
import signal
code = 137
print(code - 128, signal.Signals(code - 128).name)
print(143 - 128, signal.Signals(143 - 128).name)

9 SIGKILL
15 SIGTERM


In [ ]:
import resource, errno, tempfile, os
resource.setrlimit(resource.RLIMIT_NOFILE, (40, 40))
d = tempfile.mkdtemp()
handles = []
try:
    for i in range(100):
        handles.append(open(os.path.join(d, f"f{i}"), "w"))
except OSError as e:
    print("failed with", errno.errorcode[e.errno], "after opening between 30 and 40 files:", 30 <= len(handles) <= 40)
finally:
    for h in handles: h.close()

failed with EMFILE after opening between 30 and 40 files: True


## 5.7 Python Concurrency in Practice: Threads, Processes and asyncio

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fake_call(i):
    time.sleep(0.2)
    return i * i

t = time.perf_counter()
seq = [fake_call(i) for i in range(4)]
t_seq = time.perf_counter() - t

t = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    par = list(pool.map(fake_call, range(4)))
t_thr = time.perf_counter() - t

print(seq == par, seq)
print("sequential at least 0.8 s:", t_seq >= 0.8)
print("threads under 0.4 s:", t_thr < 0.4)

True [0, 1, 4, 9]
sequential at least 0.8 s: True
threads under 0.4 s: True


In [ ]:
import time, os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def burn(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

def timed(fn):
    t = time.perf_counter(); fn(); return time.perf_counter() - t

if __name__ == "__main__":
    N = 20_000_000
    jobs = [N] * 4
    t_seq = timed(lambda: [burn(n) for n in jobs])
    with ThreadPoolExecutor(4) as tp:
        t_thr = timed(lambda: list(tp.map(burn, jobs)))
    with ProcessPoolExecutor(4) as pp:
        t_proc = timed(lambda: list(pp.map(burn, jobs)))
    print("threads save less than 40 percent:", t_thr > 0.6 * t_seq)
    print("processes beat sequential:", t_proc < t_seq, "(needs 2+ cores:", (os.cpu_count() or 1) >= 2, ")")

threads save less than 40 percent: True
processes beat sequential: True (needs 2+ cores: True )


In [ ]:
import asyncio, time

async def fetch(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"

async def main():
    t = time.perf_counter()
    results = await asyncio.gather(fetch("a", 0.3), fetch("b", 0.1), fetch("c", 0.2), fetch("d", 0.2))
    return results, time.perf_counter() - t

results, elapsed = asyncio.run(main())
print(results)
print("elapsed about the longest wait (0.3 s), not the sum (0.8 s):", 0.3 <= elapsed < 0.5)

['a done', 'b done', 'c done', 'd done']
elapsed about the longest wait (0.3 s), not the sum (0.8 s): True


## Chapter recap

## Practice set

**1.** What is a context switch? **Answer:** The OS saves one thread's state and loads another's so a core can run it.

In [ ]:
lock.acquire()
total += compute()
lock.release()

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch05.html